# 03 — Experiment Comparison (Prompt / Embedding / Chunk)

Loads the 3 experiment reports from `app/experiments/reports/` and renders side-by-side comparison plots.

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / 'app').exists() is False:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from app.analytics.visualizer import plot_metric_comparison_bar

REPORTS_DIR = REPO_ROOT / 'app/experiments/reports'
scenarios = ['prompt_comparison', 'embedding_comparison', 'chunk_comparison']
loaded = {s: json.loads((REPORTS_DIR / f'{s}_report.json').read_text(encoding='utf-8')) for s in scenarios}
print('Loaded experiments:', list(loaded.keys()))

Loaded experiments: ['prompt_comparison', 'embedding_comparison', 'chunk_comparison']


## 1. Winners and high-level deltas

In [2]:
for s, payload in loaded.items():
    print(f'\n=== {s} ===')
    print('Winner:', payload['winner'])
    for metric, delta in payload['deltas'].items():
        sig = payload['significance'].get(metric, {})
        p_value = sig.get('p_value')
        p_disp = f'{p_value:.4f}' if isinstance(p_value, (int, float)) else 'n/a'
        print(f'  {metric}: Δ={delta["delta"]:+.4f} ({delta["relative_change"]*100:+.2f}%) p={p_disp}')


=== prompt_comparison ===
Winner: B
  generation_time: Δ=-1.5439 (-34.97%) p=0.0000
  content_length: Δ=-1049.0000 (-35.04%) p=0.0000
  section_coverage: Δ=+0.0410 (+4.81%) p=0.6985

=== embedding_comparison ===
Winner: B
  hit_at_3: Δ=+0.0356 (+4.62%) p=0.3585
  mrr: Δ=+0.0747 (+10.69%) p=0.0000
  retrieval_time: Δ=+0.0781 (+35.64%) p=0.0000

=== chunk_comparison ===
Winner: A
  chunk_count: Δ=+12.0000 (+66.67%) p=0.0000
  hit_at_3: Δ=+0.0056 (+0.71%) p=0.1581
  indexing_time: Δ=+1.1838 (+31.23%) p=0.0000


## 2. Bar comparison per experiment

In [3]:
for s, payload in loaded.items():
    variants = {v['variant']: v['metrics'] for v in payload['variants']}
    fig = plot_metric_comparison_bar(variants, title=f'{s} — Variant A vs B')
    fig

## 3. Significance heatmap

In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
rows = []
for s, payload in loaded.items():
    for metric, sig in payload['significance'].items():
        rows.append({'experiment': s, 'metric': metric, 'p_value': sig.get('p_value', 1.0) if isinstance(sig.get('p_value'), (int,float)) else 1.0})
df = pd.DataFrame(rows)
pivot = df.pivot(index='experiment', columns='metric', values='p_value').fillna(1.0)
fig, ax = plt.subplots(figsize=(8,3))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r', vmin=0, vmax=0.5, ax=ax, cbar_kws={'label': 'p-value'})
ax.set_title('Welch t-test p-values per experiment × metric (green = significant)')
fig

<Figure size 800x300 with 2 Axes>

## 4. Decision summary

Based on the synthetic executor results:
- **Prompt**: compact 8-section template wins on speed and token cost.
- **Embedding**: large model wins on accuracy at a latency cost.
- **Chunk**: 800/100 baseline is preferred for the current corpus.

Plug in the real `variant_fn` (see `docs/EXPERIMENT_GUIDE.md`) to validate against live data.